# H2 Molecule in STO-3G Basis Set

We'll compute the ground state energy of the H2 molecule using the STO-3G basis set. This is a minimal basis set that provides a simple approximation to the electronic structure of molecules.

## Constructing the Hamiltonian

First, we'll construct the Hamiltonian using PySCF.

In [40]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from propaq.datatypes import MajoranaTermSum
from scipy.optimize import differential_evolution, minimize

In [41]:
driver = PySCFDriver(atom='H 0 0 0; H 0 0 0.735', basis='sto-3g')
problem = driver.run()

In [42]:
hamiltonian = JordanWignerMapper().map(problem.hamiltonian.second_q_op())

In [43]:
import numpy as np

fci_electronic = np.linalg.eigvalsh(hamiltonian.to_matrix()).min().real
fci_total = fci_electronic + problem.nuclear_repulsion_energy

In [44]:
obs = MajoranaTermSum.from_sparse_pauli_op(hamiltonian)

## Building the Ansatz

In [45]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import XXPlusYYGate

def make_circuit(theta):
    qc = QuantumCircuit(4)
    qc.append(XXPlusYYGate(theta[0]), [0, 1])  
    qc.append(XXPlusYYGate(theta[1]), [2, 3])  
    qc.cp(theta[2], 0, 2)                       
    qc.cp(theta[3], 1, 3)                      
    qc.append(XXPlusYYGate(theta[4]), [0, 1])
    qc.append(XXPlusYYGate(theta[5]), [2, 3])
    return qc

In [46]:
from propaq.circuits import MajoranaCircuit
from propaq.propagators import MajoranaPropagator

prop = MajoranaPropagator()

HF_STATE = 0b0101

def energy(theta):
    mc = MajoranaCircuit.from_qiskit(make_circuit(theta), n_modes=8)
    return prop.expectation_value(obs, mc, HF_STATE)

## Optimization

In [47]:
import numpy as np

de_result = differential_evolution(
    energy, bounds=[(-np.pi, np.pi)] * 6, seed=42, tol=1e-6, maxiter=1000
)

result = minimize(energy, de_result.x, method='COBYLA', options={'rhobeg': 0.01, 'maxiter': 2000})

In [48]:
nuclear_repulsion = problem.nuclear_repulsion_energy
electronic = result.fun
total = electronic + nuclear_repulsion

print(f"Electronic energy: {electronic:.6f} Ha  (FCI: {fci_electronic:.6f})")
print(f"Nuclear repulsion: {nuclear_repulsion:.6f} Ha")
print(f"Total energy:      {total:.6f} Ha  (FCI: {fci_total:.6f})")

Electronic energy: -1.857275 Ha  (FCI: -1.857275)
Nuclear repulsion: 0.719969 Ha
Total energy:      -1.137306 Ha  (FCI: -1.137306)
